In [2]:
df=spark.read.table("Suppy_SilverLakehouse.dbo.cleaned_inventory_supplychain_dataset")
display(df)


StatementMeta(, 289a2326-2433-4864-8b64-dde9b455e162, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 381e44fc-0900-4eb5-8cdd-61ffe15a7925)

In [3]:
from pyspark.sql.functions import monotonically_increasing_id, col
from pyspark.sql.functions import col, year, month

# Dim_Product: Unique categories and suppliers
dim_product = df.select("category", "supplier").distinct() \
    .withColumn("product_id", monotonically_increasing_id())

# Dim_Warehouse: Unique regions and warehouses
dim_warehouse = df.select("region", "warehouse").distinct() \
    .withColumn("warehouse_id", monotonically_increasing_id())

master_date = df.select("date").distinct() \
    .withColumn("year", year(col("date"))) \
    .withColumn("month", month(col("date")))

display(master_date)  



StatementMeta(, 289a2326-2433-4864-8b64-dde9b455e162, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f28be8c8-4a8a-4e08-a669-fd16253e504b)

In [10]:
df.printSchema()

StatementMeta(, 5958fa1b-98dd-4b3f-8b02-104f1c2c9fb6, 12, Finished, Available, Finished, False)

root
 |-- date: date (nullable = true)
 |-- region: string (nullable = true)
 |-- category: string (nullable = true)
 |-- supplier: string (nullable = true)
 |-- warehouse: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- units_sold: long (nullable = true)
 |-- inventory_level: long (nullable = true)
 |-- transportation_cost: double (nullable = true)
 |-- order_accuracy: boolean (nullable = true)
 |-- lead_time__days_: long (nullable = true)
 |-- backorder: boolean (nullable = true)
 |-- COGS: double (nullable = true)
 |-- average_inventory: double (nullable = true)
 |-- warehouse_capacity: long (nullable = true)



In [4]:
fact_supplychain = df.join(dim_product, ["category", "supplier"], "left") \
    .join(dim_warehouse, ["region", "warehouse"], "left") \
    .select(
        df["date"], 
        col("product_id"), 
        col("warehouse_id"), 
        df["order_status"], 
        df["units_sold"], 
        df["inventory_level"], 
        df["transportation_cost"], 
        df["order_accuracy"], 
        df["lead_time__days_"],    # Double underscore check
        df["backorder"], 
        df["COGS"], 
        df["average_inventory"], 
        df["warehouse_capacity"]  # CHANGED TO DOUBLE UNDERSCORE
    )

display(fact_supplychain.limit(5))


StatementMeta(, 289a2326-2433-4864-8b64-dde9b455e162, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, df461dc1-2df6-4d1f-b98c-3f444f800898)

In [9]:
# Save tables to the Lakehouse visible in your sidebar
dim_product.write.format("delta").mode("overwrite").saveAsTable("dim_product")
dim_warehouse.write.format("delta").mode("overwrite").saveAsTable("dim_warehouse")
fact_supplychain.write.format("delta").mode("overwrite").saveAsTable("fact_supplychain")

print("Star Schema successfully created in G_Lakehouse!")


StatementMeta(, 289a2326-2433-4864-8b64-dde9b455e162, 11, Finished, Available, Finished, False)

Star Schema successfully created in G_Lakehouse!


In [10]:
# Example for your master_date table
master_date.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("master_date")

StatementMeta(, 289a2326-2433-4864-8b64-dde9b455e162, 12, Finished, Available, Finished, False)